# Deep Computer Vision Using Convolutional Neural Networks - Part 1

## Introduction to CNNs

While computers could beat chess champions in 1996, recognizing simple objects like puppies in images remained challenging until recently. This is because **human perception occurs unconsciously** in specialized brain modules.

**Convolutional Neural Networks (CNNs)** emerged from studying the brain's visual cortex and have achieved superhuman performance on complex visual tasks. Today, CNNs power:
- Image search services
- Self-driving cars
- Video classification systems
- Face recognition
- And much more!

In this notebook, we'll explore how CNNs work and implement them using TensorFlow and Keras.

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import load_sample_image

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

## The Architecture of the Visual Cortex

CNNs are inspired by groundbreaking neuroscience research. In 1958-1959, **Hubel and Wiesel** conducted experiments on cats that revealed key insights about how the visual cortex works:

### Key Discoveries:

1. **Local Receptive Fields**: Many neurons have small local receptive fields - they respond only to stimuli in limited visual regions

2. **Overlapping Coverage**: Receptive fields of different neurons overlap and tile the entire visual field

3. **Feature Detection**: Some neurons react only to specific patterns:
   - Horizontal lines
   - Vertical lines
   - Lines at specific angles

4. **Hierarchical Structure**: Higher-level neurons have larger receptive fields and detect complex patterns by combining lower-level patterns

This hierarchical architecture enables detection of complex patterns across the entire visual field.

### From Biology to AI:

- **1980**: The **neocognitron** was created, inspired by these findings
- **1998**: **LeNet-5** architecture by Yann LeCun introduced two crucial building blocks:
  - **Convolutional layers**
  - **Pooling layers**

### Why Not Fully Connected DNNs?

Consider a 100×100 pixel image:
- With a first layer of 1,000 neurons
- Requires **10 million connections** (100 × 100 × 1,000)
- This leads to:
  - Too many parameters
  - High memory requirements
  - Slow training
  - Overfitting

**CNNs solve this** using:
- Partially connected layers (local receptive fields)
- Weight sharing (same filter across the image)

In [ ]:
# Let's visualize the parameter explosion problem
image_size = 100 * 100  # 100x100 pixels
first_layer_neurons = 1000

# Fully connected approach
fc_parameters = image_size * first_layer_neurons
print(f"Fully Connected Network:")
print(f"  Parameters needed: {fc_parameters:,}")
print(f"  Memory (32-bit floats): {fc_parameters * 4 / (1024**2):.2f} MB")

# CNN approach (example: 5x5 filter with 1000 feature maps)
filter_size = 5 * 5
cnn_parameters = filter_size * first_layer_neurons
print(f"\nConvolutional Network:")
print(f"  Parameters needed: {cnn_parameters:,}")
print(f"  Memory (32-bit floats): {cnn_parameters * 4 / (1024**2):.2f} MB")
print(f"\nParameter reduction: {fc_parameters / cnn_parameters:.1f}x fewer parameters!")

## Convolutional Layers

Convolutional layers are the core building blocks of CNNs. They solve the parameter explosion problem while capturing spatial relationships in images.

### Basic Architecture

In convolutional layers:
- Neurons connect **only to pixels in their receptive fields** (not to every pixel)
- This architecture:
  - Concentrates on small low-level features in early layers
  - Assembles them into larger high-level features in subsequent layers
  - Matches the hierarchical structure of real-world images

**Important difference from fully connected layers:**
- Each layer is represented in **2D** (not flattened)
- Makes it easier to match neurons with their inputs
- Preserves spatial relationships

### Key Concepts

#### 1. Receptive Fields

A neuron at position (i, j) in a convolutional layer connects to neurons in the previous layer at:
- Rows: i to i + f_h - 1
- Columns: j to j + f_w - 1

Where f_h and f_w are the receptive field height and width.

#### 2. Zero Padding

Adding zeros around inputs to:
- Maintain layer dimensions
- Prevent information loss at borders
- Labeled as **"same"** padding in TensorFlow

#### 3. Stride

The shift between receptive fields:
- Stride of 1: Move one pixel at a time (dense coverage)
- Stride of 2: Skip every other pixel (reduce computational complexity)

A neuron at position (i, j) with stride (s_h, s_w) connects to inputs at:
- Rows: i × s_h to i × s_h + f_h - 1
- Columns: j × s_w to j × s_w + f_w - 1

In [ ]:
# Let's visualize receptive fields
def visualize_receptive_field():
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Create a simple input image
    input_image = np.random.rand(8, 8)
    
    # Receptive field examples
    configs = [
        {"size": 3, "stride": 1, "title": "3x3 Filter, Stride 1"},
        {"size": 3, "stride": 2, "title": "3x3 Filter, Stride 2"},
        {"size": 5, "stride": 1, "title": "5x5 Filter, Stride 1"}
    ]
    
    for idx, config in enumerate(configs):
        ax = axes[idx]
        ax.imshow(input_image, cmap='gray', alpha=0.3)
        
        # Draw receptive field for neuron at position (1, 1)
        size = config["size"]
        stride = config["stride"]
        
        # Calculate receptive field position
        i, j = 1, 1
        row_start = i * stride
        col_start = j * stride
        
        # Highlight receptive field
        from matplotlib.patches import Rectangle
        rect = Rectangle((col_start - 0.5, row_start - 0.5), size, size, 
                        linewidth=2, edgecolor='red', facecolor='red', alpha=0.3)
        ax.add_patch(rect)
        
        ax.set_title(config["title"])
        ax.set_xlabel(f"Output neuron at (1,1)\ncovers input pixels [{row_start}:{row_start+size}, {col_start}:{col_start+size}]")
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

visualize_receptive_field()

### Filters (Convolution Kernels)

**Filters** are small weight matrices the size of receptive fields. They are the learnable parameters in convolutional layers.

#### How Filters Work:

1. A filter slides across the input image
2. At each position, it computes the element-wise product with the input
3. Sums the results to produce one output value
4. This is called **convolution**

#### Example Filters:

- **Vertical Line Filter**: 7×7 matrix with 1s in the central column, 0s elsewhere
  - Enhances vertical lines in the image
  
- **Horizontal Line Filter**: 7×7 matrix with 1s in the central row, 0s elsewhere
  - Enhances horizontal lines in the image

When all neurons in a layer use the same filter, they produce a **feature map** highlighting areas that activate the filter most.

**Key insight**: During training, convolutional layers automatically learn the most useful filters for the task!

In [ ]:
# Let's create and visualize some basic filters
def create_filters():
    # Vertical line detector (7x7)
    vertical_filter = np.zeros((7, 7))
    vertical_filter[:, 3] = 1  # Middle column
    
    # Horizontal line detector (7x7)
    horizontal_filter = np.zeros((7, 7))
    horizontal_filter[3, :] = 1  # Middle row
    
    # Visualize filters
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    
    axes[0].imshow(vertical_filter, cmap='gray')
    axes[0].set_title('Vertical Line Filter')
    axes[0].axis('off')
    
    axes[1].imshow(horizontal_filter, cmap='gray')
    axes[1].set_title('Horizontal Line Filter')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return vertical_filter, horizontal_filter

vertical_filter, horizontal_filter = create_filters()

### TensorFlow Implementation

Now let's apply these filters to real images using TensorFlow!

In [ ]:
# Load sample images from scikit-learn
china = load_sample_image("china.jpg") / 255.0
flower = load_sample_image("flower.jpg") / 255.0

print(f"China image shape: {china.shape}")
print(f"Flower image shape: {flower.shape}")

# Visualize original images
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(china)
axes[0].set_title('China Image')
axes[0].axis('off')

axes[1].imshow(flower)
axes[1].set_title('Flower Image')
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Prepare images for TensorFlow (batch format)
images = np.array([china, flower])
batch_size, height, width, channels = images.shape
print(f"Batch shape: {images.shape}")
print(f"  - Batch size: {batch_size}")
print(f"  - Height: {height}")
print(f"  - Width: {width}")
print(f"  - Channels (RGB): {channels}")

In [ ]:
# Create filters for TensorFlow
# Shape: (height, width, input_channels, output_channels)
filters = np.zeros(shape=(7, 7, channels, 2), dtype=np.float32)

# Filter 0: Vertical line detector (applied to all color channels)
filters[:, 3, :, 0] = 1  # Middle column, all input channels, filter 0

# Filter 1: Horizontal line detector (applied to all color channels)
filters[3, :, :, 1] = 1  # Middle row, all input channels, filter 1

print(f"Filters shape: {filters.shape}")
print(f"  - Filter height: 7")
print(f"  - Filter width: 7")
print(f"  - Input channels: {channels}")
print(f"  - Number of filters (output feature maps): 2")

In [ ]:
# Apply convolution using tf.nn.conv2d
outputs = tf.nn.conv2d(images, filters, strides=1, padding="SAME")

print(f"Output shape: {outputs.shape}")
print(f"  - Batch size: {outputs.shape[0]}")
print(f"  - Height: {outputs.shape[1]} (same as input due to padding='SAME')")
print(f"  - Width: {outputs.shape[2]} (same as input due to padding='SAME')")
print(f"  - Feature maps: {outputs.shape[3]}")

In [ ]:
# Visualize the results
def plot_image(image):
    """Helper function to display an image"""
    if image.shape[-1] == 1:  # Grayscale
        plt.imshow(image[:, :, 0], cmap='gray')
    else:  # Color
        plt.imshow(image)
    plt.axis('off')

# Plot original images and their feature maps
fig = plt.figure(figsize=(15, 10))

for image_idx in range(2):
    # Original image
    plt.subplot(2, 3, image_idx * 3 + 1)
    plt.imshow(images[image_idx])
    plt.title(f"Original Image {image_idx + 1}")
    plt.axis('off')
    
    # Vertical line feature map
    plt.subplot(2, 3, image_idx * 3 + 2)
    plt.imshow(outputs[image_idx, :, :, 0], cmap='gray')
    plt.title(f"Vertical Lines (Filter 0)")
    plt.axis('off')
    
    # Horizontal line feature map
    plt.subplot(2, 3, image_idx * 3 + 3)
    plt.imshow(outputs[image_idx, :, :, 1], cmap='gray')
    plt.title(f"Horizontal Lines (Filter 1)")
    plt.axis('off')

plt.tight_layout()
plt.show()

print("\nNotice how:")
print("  - Filter 0 (vertical) highlights vertical edges in the images")
print("  - Filter 1 (horizontal) highlights horizontal edges in the images")
print("  - Brighter areas indicate stronger activation of the filter")

### Understanding Padding Options

TensorFlow provides two main padding options:

1. **"SAME" padding**: 
   - Adds zero padding around the input
   - Output size = ⌈input size / stride⌉
   - Preserves spatial dimensions when stride=1

2. **"VALID" padding**: 
   - No padding added
   - May ignore some rows/columns at borders
   - Output size = ⌈(input size - filter size + 1) / stride⌉

In [ ]:
# Compare SAME vs VALID padding
outputs_same = tf.nn.conv2d(images, filters, strides=1, padding="SAME")
outputs_valid = tf.nn.conv2d(images, filters, strides=1, padding="VALID")

print("Padding Comparison:")
print(f"\nInput shape: {images.shape}")
print(f"\nWith 'SAME' padding:")
print(f"  Output shape: {outputs_same.shape}")
print(f"  Height preserved: {outputs_same.shape[1] == images.shape[1]}")
print(f"  Width preserved: {outputs_same.shape[2] == images.shape[2]}")

print(f"\nWith 'VALID' padding:")
print(f"  Output shape: {outputs_valid.shape}")
print(f"  Height reduced by: {images.shape[1] - outputs_valid.shape[1]} pixels")
print(f"  Width reduced by: {images.shape[2] - outputs_valid.shape[2]} pixels")
print(f"  Reduction = filter_size - 1 = 7 - 1 = 6 pixels")

### Using Keras Conv2D Layer

In practice, we use Keras's high-level API instead of the low-level `tf.nn.conv2d`. The `Conv2D` layer handles:
- Random weight initialization
- Bias terms
- Activation functions
- And more!

In [ ]:
# Create a Conv2D layer
conv_layer = keras.layers.Conv2D(
    filters=32,           # Number of filters (output feature maps)
    kernel_size=3,        # 3x3 filter
    strides=1,            # Stride of 1
    padding="same",       # Use zero padding
    activation="relu"     # ReLU activation function
)

print("Conv2D Layer Configuration:")
print(f"  - Filters: 32")
print(f"  - Kernel size: 3x3")
print(f"  - Stride: 1")
print(f"  - Padding: 'same'")
print(f"  - Activation: ReLU")

# Apply to our images
feature_maps = conv_layer(images)
print(f"\nOutput shape: {feature_maps.shape}")

In [ ]:
# Visualize some of the learned feature maps
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
fig.suptitle('32 Feature Maps from Conv2D Layer (China Image)', fontsize=14)

for i in range(32):
    row = i // 8
    col = i % 8
    axes[row, col].imshow(feature_maps[0, :, :, i], cmap='viridis')
    axes[row, col].set_title(f'Map {i}', fontsize=8)
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

print("Each feature map highlights different patterns in the image!")
print("The filters were randomly initialized, so patterns vary.")

### Stacking Multiple Feature Maps

A convolutional layer actually outputs **multiple feature maps** in 3D:

- **One feature map per filter**
- **One neuron per pixel** in each feature map
- **All neurons within a feature map share the same parameters** (dramatically reducing model parameters)
- **Neurons in different feature maps use different parameters**

#### Key Advantage: Translation Invariance

Once the CNN learns to recognize a pattern in one location, **it can recognize it anywhere**!

Regular DNNs must relearn patterns for each location, but CNNs use the same filter across the entire image.

In [ ]:
# Demonstrate parameter sharing
print("Parameter Sharing in CNNs:\n")

# Example configuration
input_height, input_width = 28, 28
kernel_size = 3
num_filters = 32
input_channels = 1  # Grayscale

# Calculate parameters
# Each filter has: kernel_size * kernel_size * input_channels weights + 1 bias
params_per_filter = (kernel_size * kernel_size * input_channels) + 1
total_params = params_per_filter * num_filters

# Output size (with same padding)
output_height, output_width = input_height, input_width
total_neurons = output_height * output_width * num_filters

print(f"Configuration:")
print(f"  Input: {input_height}x{input_width}x{input_channels}")
print(f"  Kernel: {kernel_size}x{kernel_size}")
print(f"  Filters: {num_filters}")
print(f"\nParameters:")
print(f"  Per filter: {params_per_filter} ({kernel_size}x{kernel_size}x{input_channels} + 1 bias)")
print(f"  Total: {total_params:,}")
print(f"\nNeurons:")
print(f"  Per feature map: {output_height * output_width:,}")
print(f"  Total: {total_neurons:,}")
print(f"\nParameter Sharing Ratio:")
print(f"  {total_neurons:,} neurons share only {total_params:,} parameters")
print(f"  Ratio: {total_neurons / total_params:.1f}:1")

# Compare to fully connected
fc_params = (input_height * input_width * input_channels) * (output_height * output_width * num_filters)
print(f"\nIf fully connected: {fc_params:,} parameters")
print(f"Reduction: {fc_params / total_params:.0f}x fewer parameters!")

### Mathematical Formulation

The output of a neuron in a convolutional layer is computed as:

$$z_{i,j,k} = b_k + \sum_{u=0}^{f_h-1} \sum_{v=0}^{f_w-1} \sum_{k'=0}^{f_n'-1} x_{i',j',k'} \cdot w_{u,v,k',k}$$

where:
- $i' = i \times s_h + u$
- $j' = j \times s_w + v$

**Variables:**
- $z_{i,j,k}$: Output of neuron at row $i$, column $j$ in feature map $k$
- $s_h, s_w$: Vertical and horizontal strides
- $f_h, f_w$: Receptive field height and width
- $f_n'$: Number of feature maps in previous layer
- $x_{i',j',k'}$: Input from previous layer at position $(i', j', k')$
- $b_k$: Bias term for feature map $k$
- $w_{u,v,k',k}$: Connection weight

This is exactly what TensorFlow computes for us!

In [ ]:
# Let's manually verify the convolution formula with a simple example
def manual_convolution_2d(image, kernel, stride=1):
    """
    Simple 2D convolution implementation (single channel, no padding)
    to demonstrate the mathematical formula
    """
    img_h, img_w = image.shape
    ker_h, ker_w = kernel.shape
    
    # Calculate output dimensions
    out_h = (img_h - ker_h) // stride + 1
    out_w = (img_w - ker_w) // stride + 1
    
    output = np.zeros((out_h, out_w))
    
    # Apply convolution
    for i in range(out_h):
        for j in range(out_w):
            # Calculate receptive field position
            i_start = i * stride
            j_start = j * stride
            
            # Extract receptive field
            receptive_field = image[i_start:i_start+ker_h, j_start:j_start+ker_w]
            
            # Compute convolution: element-wise multiply and sum
            output[i, j] = np.sum(receptive_field * kernel)
    
    return output

# Test with a simple example
test_image = np.array([
    [1, 2, 3, 4, 5],
    [5, 4, 3, 2, 1],
    [1, 2, 3, 4, 5],
    [5, 4, 3, 2, 1],
    [1, 2, 3, 4, 5]
], dtype=np.float32)

test_kernel = np.array([
    [0, 1, 0],
    [0, 1, 0],
    [0, 1, 0]
], dtype=np.float32)  # Vertical line detector

# Manual computation
manual_result = manual_convolution_2d(test_image, test_kernel, stride=1)

print("Test Image:")
print(test_image)
print("\nKernel (Vertical Line Detector):")
print(test_kernel)
print("\nManual Convolution Result:")
print(manual_result)

# Verify with TensorFlow
tf_image = test_image.reshape(1, 5, 5, 1)  # Add batch and channel dimensions
tf_kernel = test_kernel.reshape(3, 3, 1, 1)  # Add input/output channel dimensions
tf_result = tf.nn.conv2d(tf_image, tf_kernel, strides=1, padding="VALID")

print("\nTensorFlow Result:")
print(tf_result[0, :, :, 0].numpy())
print("\nResults match:", np.allclose(manual_result, tf_result[0, :, :, 0].numpy()))

### Memory Requirements

CNNs require significant RAM, especially during training. Understanding memory requirements helps avoid out-of-memory errors.

#### Example Calculation:

A layer with:
- 5×5 filters
- Outputting 200 feature maps
- Of size 150×100
- Applied to RGB images (3 channels)

**Parameters:**
- Per filter: (5 × 5 × 3 + 1) = 76 parameters
- Total: 76 × 200 = **15,200 parameters**

**Computations:**
- Per output pixel: 5 × 5 × 3 = 75 multiplications
- Total: 75 × 150 × 100 × 200 = **225 million multiplications**

**Memory (32-bit floats):**
- Output: 150 × 100 × 200 × 4 bytes = **12 MB per instance**
- Batch of 100: **1.2 GB**

In [ ]:
# Calculate memory requirements for different CNN configurations
def calculate_memory_requirements(input_h, input_w, input_channels, 
                                 kernel_size, num_filters, stride, 
                                 batch_size, dtype_bytes=4):
    """
    Calculate memory requirements for a convolutional layer
    """
    # Parameters
    params_per_filter = (kernel_size * kernel_size * input_channels) + 1
    total_params = params_per_filter * num_filters
    
    # Output dimensions (with 'same' padding)
    output_h = input_h // stride
    output_w = input_w // stride
    
    # Computations
    computations_per_pixel = kernel_size * kernel_size * input_channels
    total_computations = computations_per_pixel * output_h * output_w * num_filters
    
    # Memory
    memory_per_instance = output_h * output_w * num_filters * dtype_bytes
    memory_batch = memory_per_instance * batch_size
    
    return {
        'params': total_params,
        'computations': total_computations,
        'memory_per_instance_mb': memory_per_instance / (1024**2),
        'memory_batch_mb': memory_batch / (1024**2),
        'output_shape': (batch_size, output_h, output_w, num_filters)
    }

# Example from the text
print("Example: Large Convolutional Layer")
print("="*50)
results = calculate_memory_requirements(
    input_h=150, input_w=100, input_channels=3,
    kernel_size=5, num_filters=200, stride=1, batch_size=100
)

print(f"Parameters: {results['params']:,}")
print(f"Computations: {results['computations'] / 1e6:.0f} million")
print(f"Memory per instance: {results['memory_per_instance_mb']:.2f} MB")
print(f"Memory for batch of 100: {results['memory_batch_mb']:.2f} MB")
print(f"Output shape: {results['output_shape']}")

# More practical example
print("\n\nPractical Example: Standard CNN Layer")
print("="*50)
results2 = calculate_memory_requirements(
    input_h=224, input_w=224, input_channels=3,
    kernel_size=3, num_filters=64, stride=1, batch_size=32
)

print(f"Parameters: {results2['params']:,}")
print(f"Computations: {results2['computations'] / 1e6:.0f} million")
print(f"Memory per instance: {results2['memory_per_instance_mb']:.2f} MB")
print(f"Memory for batch of 32: {results2['memory_batch_mb']:.2f} MB")
print(f"Output shape: {results2['output_shape']}")

### Solutions for Out-of-Memory Errors

If your GPU runs out of memory:

1. **Reduce mini-batch size** - Most effective but may affect training dynamics
2. **Increase stride** - Reduces dimensionality but may lose information
3. **Remove layers** - Simplifies model but may reduce accuracy
4. **Use 16-bit floats** instead of 32-bit - Halves memory usage
5. **Distribute across multiple devices** - Requires more resources

Let's see the impact of these strategies:

In [ ]:
# Compare memory usage with different strategies
base_config = {
    'input_h': 224, 'input_w': 224, 'input_channels': 3,
    'kernel_size': 3, 'num_filters': 64, 'stride': 1, 'batch_size': 32
}

print("Memory Optimization Strategies")
print("="*60)

# Baseline
base = calculate_memory_requirements(**base_config)
print(f"\n1. BASELINE (32-bit floats, batch=32)")
print(f"   Memory: {base['memory_batch_mb']:.2f} MB")

# Strategy 1: Reduce batch size
config1 = base_config.copy()
config1['batch_size'] = 16
result1 = calculate_memory_requirements(**config1)
print(f"\n2. Reduce Batch Size (32 → 16)")
print(f"   Memory: {result1['memory_batch_mb']:.2f} MB")
print(f"   Reduction: {(1 - result1['memory_batch_mb']/base['memory_batch_mb'])*100:.0f}%")

# Strategy 2: Increase stride
config2 = base_config.copy()
config2['stride'] = 2
result2 = calculate_memory_requirements(**config2)
print(f"\n3. Increase Stride (1 → 2)")
print(f"   Memory: {result2['memory_batch_mb']:.2f} MB")
print(f"   Reduction: {(1 - result2['memory_batch_mb']/base['memory_batch_mb'])*100:.0f}%")
print(f"   Output shape: {result2['output_shape']}")

# Strategy 3: Reduce filters
config3 = base_config.copy()
config3['num_filters'] = 32
result3 = calculate_memory_requirements(**config3)
print(f"\n4. Reduce Filters (64 → 32)")
print(f"   Memory: {result3['memory_batch_mb']:.2f} MB")
print(f"   Reduction: {(1 - result3['memory_batch_mb']/base['memory_batch_mb'])*100:.0f}%")

# Strategy 4: Use 16-bit floats
config4 = base_config.copy()
result4 = calculate_memory_requirements(**config4, dtype_bytes=2)
print(f"\n5. Use 16-bit Floats (32-bit → 16-bit)")
print(f"   Memory: {result4['memory_batch_mb']:.2f} MB")
print(f"   Reduction: {(1 - result4['memory_batch_mb']/base['memory_batch_mb'])*100:.0f}%")

# Combined strategies
config5 = base_config.copy()
config5['batch_size'] = 16
config5['stride'] = 2
result5 = calculate_memory_requirements(**config5, dtype_bytes=2)
print(f"\n6. Combined (batch↓, stride↑, 16-bit)")
print(f"   Memory: {result5['memory_batch_mb']:.2f} MB")
print(f"   Reduction: {(1 - result5['memory_batch_mb']/base['memory_batch_mb'])*100:.0f}%")

## Summary of Part 1

In this first part, we covered:

### 1. Introduction to CNNs
- CNNs emerged from studying the brain's visual cortex
- They solve the parameter explosion problem of fully connected networks
- Used in image search, self-driving cars, and more

### 2. Visual Cortex Architecture
- Local receptive fields
- Hierarchical feature detection
- From neocognitron (1980) to LeNet-5 (1998)

### 3. Convolutional Layers - The Core Building Block
- **Receptive fields**: Local connectivity
- **Padding**: Zero padding to preserve dimensions
- **Stride**: Control spatial resolution and computation
- **Filters**: Learnable feature detectors
- **Feature maps**: Multiple filters create multiple feature maps
- **Parameter sharing**: Same filter across entire image
- **Translation invariance**: Recognize patterns anywhere

### 4. Implementation
- Low-level: `tf.nn.conv2d()`
- High-level: `keras.layers.Conv2D()`
- Padding options: "SAME" vs "VALID"

### 5. Memory Requirements
- Understanding parameter counts
- Computational complexity
- Memory usage per batch
- Optimization strategies

### Next Steps

In the next part, we'll cover:
- **Pooling Layers**: Downsampling and invariance
- **CNN Architectures**: LeNet-5, AlexNet, VGGNet, ResNet, and more
- **Practical Applications**: Building real CNNs for classification tasks

In [ ]:
# Quick reference: Creating a simple CNN layer stack
print("Quick Reference: Simple CNN Stack\n")
print("Code:")
print("""
model = keras.Sequential([
    # First convolutional block
    keras.layers.Conv2D(32, kernel_size=3, activation='relu', 
                       padding='same', input_shape=[224, 224, 3]),
    
    # Second convolutional block
    keras.layers.Conv2D(64, kernel_size=3, activation='relu', 
                       padding='same'),
    
    # More layers to come in Part 2!
])
""")

print("\nKey Parameters:")
print("  - filters: Number of output feature maps")
print("  - kernel_size: Size of the convolution filter (3 means 3x3)")
print("  - strides: Shift between receptive fields (default=1)")
print("  - padding: 'same' or 'valid'")
print("  - activation: 'relu', 'sigmoid', 'tanh', etc.")

---

# Part 2: Pooling Layers and CNN Architectures

## Pooling Layers

Pooling layers are another crucial building block of CNNs. They subsample the input to:
- **Reduce computational load**
- **Reduce memory usage**
- **Reduce parameters** (limiting overfitting risk)

### How Pooling Layers Work

Like convolutional layers, neurons in pooling layers connect to limited receptive fields. However, pooling layers have **no weights**. Instead, they aggregate inputs using functions like:
- **Max pooling**: Takes the maximum value
- **Average pooling**: Takes the mean value

#### Max Pooling Example:
- 2×2 pooling kernel
- Stride 2
- No padding
- **Result**: Only the maximum value in each receptive field propagates forward
- **Output dimensions**: Half the height and half the width of input

**Important**: Pooling layers work independently on each input channel, so output depth equals input depth.

In [ ]:
# Demonstrate pooling with a simple example
def demonstrate_pooling():
    # Create a simple input
    sample_input = np.array([
        [1, 3, 2, 4],
        [5, 6, 1, 3],
        [2, 4, 8, 2],
        [1, 3, 5, 7]
    ], dtype=np.float32)
    
    print("Original Input (4x4):")
    print(sample_input)
    
    # Reshape for TensorFlow (add batch and channel dimensions)
    input_tensor = sample_input.reshape(1, 4, 4, 1)
    
    # Max pooling with 2x2 kernel, stride 2
    max_pooled = tf.nn.max_pool(input_tensor, ksize=2, strides=2, padding='VALID')
    
    # Average pooling with 2x2 kernel, stride 2
    avg_pooled = tf.nn.avg_pool(input_tensor, ksize=2, strides=2, padding='VALID')
    
    print("\n\nMax Pooling (2x2, stride 2):")
    print(max_pooled[0, :, :, 0].numpy())
    print("Note: Each value is the maximum from a 2x2 region")
    
    print("\n\nAverage Pooling (2x2, stride 2):")
    print(avg_pooled[0, :, :, 0].numpy())
    print("Note: Each value is the average from a 2x2 region")
    
    # Visualize the process
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].imshow(sample_input, cmap='viridis', interpolation='nearest')
    axes[0].set_title('Original Input (4×4)', fontsize=12)
    axes[0].grid(True, linewidth=2, color='white')
    for i in range(4):
        for j in range(4):
            axes[0].text(j, i, f'{sample_input[i, j]:.0f}', 
                        ha='center', va='center', color='white', fontsize=14)
    
    axes[1].imshow(max_pooled[0, :, :, 0], cmap='viridis', interpolation='nearest')
    axes[1].set_title('Max Pooling Output (2×2)', fontsize=12)
    axes[1].grid(True, linewidth=2, color='white')
    for i in range(2):
        for j in range(2):
            axes[1].text(j, i, f'{max_pooled[0, i, j, 0]:.0f}', 
                        ha='center', va='center', color='white', fontsize=14)
    
    axes[2].imshow(avg_pooled[0, :, :, 0], cmap='viridis', interpolation='nearest')
    axes[2].set_title('Average Pooling Output (2×2)', fontsize=12)
    axes[2].grid(True, linewidth=2, color='white')
    for i in range(2):
        for j in range(2):
            axes[2].text(j, i, f'{avg_pooled[0, i, j, 0]:.1f}', 
                        ha='center', va='center', color='white', fontsize=14)
    
    plt.tight_layout()
    plt.show()

demonstrate_pooling()

In [ ]:
# Apply pooling to real images
# Using the images we loaded earlier
print("Applying pooling to real images...")
print(f"Original image shape: {images.shape}")

# Max pooling
max_pool_layer = keras.layers.MaxPool2D(pool_size=2)
max_pooled_images = max_pool_layer(images)

print(f"After max pooling (2x2): {max_pooled_images.shape}")

# Visualize the effect
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].imshow(china)
axes[0, 0].set_title('Original China Image (427×640)', fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(max_pooled_images[0])
axes[0, 1].set_title('After Max Pooling (213×320)', fontsize=12)
axes[0, 1].axis('off')

axes[1, 0].imshow(flower)
axes[1, 0].set_title('Original Flower Image (427×640)', fontsize=12)
axes[1, 0].axis('off')

axes[1, 1].imshow(max_pooled_images[1])
axes[1, 1].set_title('After Max Pooling (213×320)', fontsize=12)
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

print("\nObservation: The pooled images are half the size but retain most visual information!")

### Benefits of Pooling

#### 1. Translation Invariance

Max pooling provides some **invariance to small translations**. If we shift an image by 1-2 pixels, the pooling output may be identical or nearly identical.

Let's demonstrate this:

In [ ]:
# Demonstrate translation invariance
test_pattern = np.array([
    [0, 0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0, 0],
    [0, 1, 1, 1, 0, 0],
    [0, 1, 1, 1, 0, 0],
    [0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0]
], dtype=np.float32)

# Shifted version (1 pixel right, 1 pixel down)
shifted_pattern = np.array([
    [0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0],
    [0, 0, 1, 1, 1, 0],
    [0, 0, 1, 1, 1, 0],
    [0, 0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 0]
], dtype=np.float32)

# Reshape for TensorFlow
pattern_batch = np.array([test_pattern, shifted_pattern]).reshape(2, 6, 6, 1)

# Apply max pooling
pooled = tf.nn.max_pool(pattern_batch, ksize=2, strides=2, padding='VALID')

print("Original Pattern:")
print(test_pattern)
print("\nShifted Pattern (1 pixel right and down):")
print(shifted_pattern)
print("\nMax Pooled Original:")
print(pooled[0, :, :, 0].numpy())
print("\nMax Pooled Shifted:")
print(pooled[1, :, :, 0].numpy())
print("\nOutputs are identical! This is translation invariance.")

#### 2. Other Invariances

Pooling also provides limited:
- **Rotational invariance**: Small rotations may produce similar outputs
- **Scale invariance**: Small size changes may produce similar outputs

### Downsides of Pooling

#### 1. Destructive Nature

A 2×2 pooling kernel with stride 2 **drops 75% of input values**! This is a lot of information loss.

#### 2. Not Always Desirable

For some tasks, we need **equivariance**, not invariance:
- **Equivariance**: Small input changes → corresponding output changes
- **Example**: Semantic segmentation (pixel-level classification)
- If we shift the input, we want the output to shift correspondingly

### Keras Implementation

In [ ]:
# Keras pooling layers
print("Creating Pooling Layers in Keras:\n")

# Max pooling layer
max_pool = keras.layers.MaxPool2D(pool_size=2)
print("Max Pooling Layer:")
print(f"  Pool size: 2x2")
print(f"  Default stride: 2 (same as pool_size)")

# Average pooling layer
avg_pool = keras.layers.AvgPool2D(pool_size=2)
print("\nAverage Pooling Layer:")
print(f"  Pool size: 2x2")
print(f"  Default stride: 2 (same as pool_size)")

# Global average pooling
global_avg_pool = keras.layers.GlobalAvgPool2D()
print("\nGlobal Average Pooling Layer:")
print(f"  Computes the mean of each entire feature map")
print(f"  Output: One number per feature map per instance")
print(f"  Extremely destructive but useful as output layer")

# Demonstrate global average pooling
print("\n" + "="*60)
print("Global Average Pooling Example:")
sample_feature_maps = np.random.rand(2, 8, 8, 3)  # 2 images, 8x8, 3 channels
print(f"Input shape: {sample_feature_maps.shape}")

global_pooled = global_avg_pool(sample_feature_maps)
print(f"Output shape: {global_pooled.shape}")
print(f"Reduced from 8x8x3 = 192 values to just 3 values per image!")

## CNN Architectures

Now let's look at how to combine convolutional and pooling layers into complete CNN architectures!

### Typical CNN Structure

A typical CNN follows this pattern:

1. **Few convolutional layers** (+ ReLU activation)
2. **Pooling layer**
3. **Repeat steps 1-2** multiple times
4. **Regular feedforward network** (fully connected layers + ReLU)
5. **Final output layer** (e.g., softmax for classification)

**Best practice**: 
- Use **smaller kernels (3×3) stacked** rather than large kernels (5×5)
- Exception: First layer can use larger kernels with stride ≥2

### Example: Fashion MNIST CNN

Let's build a complete CNN for the Fashion MNIST dataset (28×28 grayscale images, 10 classes).

In [ ]:
# Load Fashion MNIST dataset
fashion_mnist = keras.datasets.fashion_mnist
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

# Normalize pixel values to 0-1 range
X_train_full = X_train_full / 255.0
X_test = X_test / 255.0

# Create validation set
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]

# Reshape to add channel dimension (grayscale = 1 channel)
X_train = X_train[..., np.newaxis]
X_valid = X_valid[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print("Fashion MNIST Dataset:")
print(f"  Training set: {X_train.shape}")
print(f"  Validation set: {X_valid.shape}")
print(f"  Test set: {X_test.shape}")
print(f"  Number of classes: 10")

# Class names
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

# Visualize some examples
plt.figure(figsize=(12, 6))
for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(X_train[i, :, :, 0], cmap='gray')
    plt.title(class_names[y_train[i]], fontsize=9)
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Build CNN model for Fashion MNIST
model = keras.models.Sequential([
    # First convolutional block
    keras.layers.Conv2D(64, 7, activation="relu", padding="same",
                       input_shape=[28, 28, 1], name="conv1"),
    keras.layers.MaxPooling2D(2, name="pool1"),
    
    # Second convolutional block (2 conv layers)
    keras.layers.Conv2D(128, 3, activation="relu", padding="same", name="conv2_1"),
    keras.layers.Conv2D(128, 3, activation="relu", padding="same", name="conv2_2"),
    keras.layers.MaxPooling2D(2, name="pool2"),
    
    # Third convolutional block (2 conv layers)
    keras.layers.Conv2D(256, 3, activation="relu", padding="same", name="conv3_1"),
    keras.layers.Conv2D(256, 3, activation="relu", padding="same", name="conv3_2"),
    keras.layers.MaxPooling2D(2, name="pool3"),
    
    # Flatten and dense layers
    keras.layers.Flatten(name="flatten"),
    keras.layers.Dense(128, activation="relu", name="fc1"),
    keras.layers.Dropout(0.5, name="dropout1"),
    keras.layers.Dense(64, activation="relu", name="fc2"),
    keras.layers.Dropout(0.5, name="dropout2"),
    keras.layers.Dense(10, activation="softmax", name="output")
])

# Display model architecture
model.summary()

print("\n" + "="*70)
print("Architecture Pattern:")
print("  1. Conv(64) -> Pool  [reduces 28x28 -> 14x14]")
print("  2. Conv(128) -> Conv(128) -> Pool  [reduces 14x14 -> 7x7]")
print("  3. Conv(256) -> Conv(256) -> Pool  [reduces 7x7 -> 3x3]")
print("  4. Flatten -> Dense(128) -> Dropout -> Dense(64) -> Dropout -> Output(10)")
print("\nNote: Number of filters doubles after each pooling layer (64->128->256)")

In [ ]:
# Compile the model
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

print("Model compiled successfully!")
print("\nConfiguration:")
print(f"  Loss: Sparse Categorical Crossentropy")
print(f"  Optimizer: Adam")
print(f"  Metrics: Accuracy")

# Count parameters
total_params = model.count_params()
print(f"\nTotal parameters: {total_params:,}")

# Calculate model size
model_size_mb = (total_params * 4) / (1024**2)  # 4 bytes per float32
print(f"Model size (32-bit floats): {model_size_mb:.2f} MB")

In [ ]:
# Train the model (just a few epochs for demonstration)
print("Training the model...")
print("Note: Training on CPU may take several minutes. GPU is much faster!\n")

history = model.fit(
    X_train, y_train,
    epochs=5,
    validation_data=(X_valid, y_valid),
    batch_size=128
)

print("\nTraining completed!")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot accuracy
axes[0].plot(history.history['accuracy'], label='Training Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot loss
axes[1].plot(history.history['loss'], label='Training Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Model Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Evaluate on test set
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Accuracy: {test_accuracy*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

In [ ]:
# Make predictions on test samples
n_samples = 10
sample_indices = np.random.choice(len(X_test), n_samples, replace=False)
X_samples = X_test[sample_indices]
y_samples = y_test[sample_indices]

# Get predictions
predictions = model.predict(X_samples)
predicted_classes = np.argmax(predictions, axis=1)

# Visualize predictions
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('CNN Predictions on Fashion MNIST', fontsize=14)

for i, ax in enumerate(axes.flat):
    ax.imshow(X_samples[i, :, :, 0], cmap='gray')
    
    true_label = class_names[y_samples[i]]
    pred_label = class_names[predicted_classes[i]]
    confidence = predictions[i, predicted_classes[i]] * 100
    
    color = 'green' if y_samples[i] == predicted_classes[i] else 'red'
    ax.set_title(f'True: {true_label}\nPred: {pred_label}\n({confidence:.1f}%)', 
                fontsize=9, color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()

# Calculate accuracy for these samples
accuracy = np.mean(y_samples == predicted_classes)
print(f"\nAccuracy on these {n_samples} samples: {accuracy*100:.1f}%")

### Key Design Patterns in CNNs

From our Fashion MNIST example, notice these important patterns:

#### 1. Progressive Feature Map Growth
- Layer 1: 64 filters
- Layer 2: 128 filters (2×)
- Layer 3: 256 filters (2×)

**Why?** As spatial dimensions decrease (due to pooling), we increase the number of feature maps to maintain representational capacity.

#### 2. Spatial Dimension Reduction
- Input: 28×28
- After pool1: 14×14 (÷2)
- After pool2: 7×7 (÷2)
- After pool3: 3×3 (÷2)

**Why?** Reduces computation while building hierarchical features.

#### 3. Multiple Convolutions Between Pooling
- Stack 2-3 convolutional layers before each pooling layer
- Allows learning more complex features at each scale

#### 4. Dropout for Regularization
- Applied to dense layers (50% rate)
- Prevents overfitting
- Not typically used in convolutional layers

### Visualizing What CNNs Learn

Let's visualize the filters learned by our CNN to understand what features it detects!

In [ ]:
# Visualize filters from the first convolutional layer
conv1_weights = model.get_layer('conv1').get_weights()[0]
print(f"First layer filters shape: {conv1_weights.shape}")
print(f"  Filter size: {conv1_weights.shape[0]}x{conv1_weights.shape[1]}")
print(f"  Input channels: {conv1_weights.shape[2]}")
print(f"  Number of filters: {conv1_weights.shape[3]}")

# Normalize filters for visualization
def normalize_filter(f):
    f_min, f_max = f.min(), f.max()
    return (f - f_min) / (f_max - f_min + 1e-8)

# Plot first 64 filters
fig, axes = plt.subplots(8, 8, figsize=(12, 12))
fig.suptitle('Learned Filters in First Convolutional Layer (conv1)', fontsize=14)

for i in range(64):
    ax = axes[i // 8, i % 8]
    # Get filter (7x7x1 for grayscale)
    filter_img = conv1_weights[:, :, 0, i]
    filter_img = normalize_filter(filter_img)
    
    ax.imshow(filter_img, cmap='gray')
    ax.axis('off')
    ax.set_title(f'F{i}', fontsize=8)

plt.tight_layout()
plt.show()

print("\nThese filters detect various low-level features:")
print("  - Edges at different orientations")
print("  - Gradients")
print("  - Simple patterns")
print("  - Textural elements")

In [ ]:
# Visualize feature maps produced by each layer
# Create a model that outputs all intermediate layers
layer_outputs = [layer.output for layer in model.layers if 'conv' in layer.name or 'pool' in layer.name]
activation_model = keras.Model(inputs=model.input, outputs=layer_outputs)

# Get activations for a sample image
sample_img = X_test[0:1]  # Take first test image
activations = activation_model.predict(sample_img)

# Layer names
layer_names = [layer.name for layer in model.layers if 'conv' in layer.name or 'pool' in layer.name]

print("Visualizing feature maps from different layers...")
print(f"Input image shape: {sample_img.shape}")
print(f"\nLayer outputs:")
for name, activation in zip(layer_names, activations):
    print(f"  {name}: {activation.shape}")

# Visualize feature maps from conv1
plt.figure(figsize=(16, 4))
plt.suptitle(f'Feature Maps from conv1 (showing 16 of 64)', fontsize=14)

n_features = 16
for i in range(n_features):
    plt.subplot(2, 8, i + 1)
    plt.imshow(activations[0][0, :, :, i], cmap='viridis')
    plt.axis('off')
    plt.title(f'Map {i}', fontsize=8)

plt.tight_layout()
plt.show()

# Visualize progression through layers
fig = plt.figure(figsize=(16, 10))
fig.suptitle('Feature Maps Progression Through Network', fontsize=16)

images_per_row = 8
for layer_idx, (layer_name, layer_activation) in enumerate(zip(layer_names[:6], activations[:6])):
    n_features = layer_activation.shape[-1]
    n_to_display = min(16, n_features)
    
    for i in range(n_to_display):
        plt.subplot(6, images_per_row, layer_idx * images_per_row + i + 1)
        plt.imshow(layer_activation[0, :, :, i], cmap='viridis')
        plt.axis('off')
        if i == 0:
            plt.ylabel(layer_name, fontsize=10, rotation=0, ha='right')

plt.tight_layout()
plt.show()

print("\nObservations:")
print("  - Early layers detect simple features (edges, textures)")
print("  - Later layers detect more abstract, complex patterns")
print("  - Feature maps become smaller due to pooling")
print("  - Number of feature maps increases in deeper layers")

## Summary of Part 2

In this second part, we covered:

### 1. Pooling Layers
- **Purpose**: Reduce computational load, memory, and parameters
- **How they work**: No weights, use aggregation functions (max, average)
- **Max pooling**: Takes maximum value in receptive field
- **Average pooling**: Takes mean value
- **Translation invariance**: Small shifts produce similar outputs
- **Downside**: Destructive (drops 75% of values with 2×2 pooling)
- **Keras layers**: `MaxPool2D`, `AvgPool2D`, `GlobalAvgPool2D`

### 2. CNN Architecture Patterns
- Stack: Convolution → Pooling → Repeat → Dense → Output
- Use smaller kernels (3×3) stacked instead of large kernels
- Double feature maps after each pooling layer
- Use dropout on dense layers for regularization

### 3. Practical Implementation
- Built complete CNN for Fashion MNIST
- Achieved competitive accuracy with relatively simple architecture
- Visualized learned filters and feature maps
- Observed hierarchical feature learning:
  - Early layers: Simple features (edges, textures)
  - Later layers: Complex, abstract patterns

### Key Insights

1. **Parameter Efficiency**: CNNs use far fewer parameters than fully connected networks
2. **Hierarchical Learning**: Features become more abstract in deeper layers
3. **Translation Invariance**: Same filter applied everywhere
4. **Design Trade-offs**: 
   - More pooling = Less computation but more information loss
   - More filters = More capacity but more computation

### What's Next

In Part 3, we'll explore:
- Famous CNN architectures (LeNet-5, AlexNet, VGGNet, ResNet, etc.)
- Transfer learning and pretrained models
- Advanced techniques and applications

This notebook is now a complete, runnable tutorial for the first 50% of the README content!

---

# Part 3: Famous CNN Architectures and Advanced Techniques

## Historic CNN Architectures

Let's explore the landmark architectures that revolutionized computer vision!

### LeNet-5 (1998)

Created by **Yann LeCun** for handwritten digit recognition (MNIST dataset).

**Architecture**:
- Input: 32×32 (MNIST images zero-padded from 28×28)
- **C1**: Convolution, 6 feature maps, 28×28, 5×5 kernel, stride 1, tanh activation
- **S2**: Average pooling, 6 maps, 14×14, 2×2 kernel, stride 2, tanh activation
- **C3**: Convolution, 16 maps, 10×10, 5×5 kernel, stride 1, tanh activation
- **S4**: Average pooling, 16 maps, 5×5, 2×2 kernel, stride 2, tanh activation
- **C5**: Convolution, 120 maps, 1×1, 5×5 kernel, stride 1, tanh activation
- **F6**: Fully connected, 84 units, tanh activation
- **Output**: Fully connected, 10 units

**Key Features**:
- Used average pooling (common at the time)
- Used tanh activation (before ReLU became popular)
- Relatively small by modern standards

In [ ]:
# Implement LeNet-5 architecture
def create_lenet5():
    """Create LeNet-5 architecture (modified for 28x28 MNIST)"""
    model = keras.models.Sequential([
        # C1: Convolution
        keras.layers.Conv2D(6, kernel_size=5, strides=1, activation='tanh', 
                           padding='same', input_shape=[28, 28, 1]),
        # S2: Average pooling
        keras.layers.AvgPool2D(pool_size=2),
        
        # C3: Convolution
        keras.layers.Conv2D(16, kernel_size=5, strides=1, activation='tanh', 
                           padding='valid'),
        # S4: Average pooling
        keras.layers.AvgPool2D(pool_size=2),
        
        # C5: Convolution (acts like dense layer with 5x5 input)
        keras.layers.Conv2D(120, kernel_size=5, strides=1, activation='tanh', 
                           padding='valid'),
        
        # Flatten
        keras.layers.Flatten(),
        
        # F6: Fully connected
        keras.layers.Dense(84, activation='tanh'),
        
        # Output
        keras.layers.Dense(10, activation='softmax')
    ], name='LeNet-5')
    
    return model

lenet5 = create_lenet5()
lenet5.summary()

print("\nLeNet-5 Characteristics:")
print("  - One of the first CNNs for practical use")
print("  - Used tanh activation (before ReLU)")
print("  - Used average pooling (before max pooling became standard)")
print("  - Relatively few parameters by today's standards")
print(f"  - Total parameters: {lenet5.count_params():,}")

### AlexNet (2012)

**Won ILSVRC 2012** with 17% top-5 error rate (vs. 26% for second place) - a groundbreaking achievement!

**Key Innovations**:

1. **Much larger and deeper** than LeNet-5
2. **First to stack convolutional layers directly** without pooling between them
3. **Used ReLU activation** instead of tanh (much faster training)
4. **Used dropout** (50% rate) on fully connected layers
5. **Data augmentation**: Random shifts, horizontal flips, lighting changes
6. **Local Response Normalization (LRN)**: Strongly activated neurons inhibit others in neighboring feature maps

**Architecture Highlights**:
- Input: 227×227×3 (RGB images)
- 5 Convolutional layers
- 3 Fully connected layers
- ~60 million parameters

### Local Response Normalization (LRN)

LRN formula:

$$b_i = \frac{a_i}{\left(k + \alpha \sum_{j=j_{low}}^{j_{high}} a_j^2\right)^\beta}$$

where:
- $j_{high} = \min(i + r/2, f_n - 1)$
- $j_{low} = \max(0, i - r/2)$
- Typical parameters: $r=2$, $\alpha=0.00002$, $\beta=0.75$, $k=1$

**Note**: LRN is rarely used today; Batch Normalization has largely replaced it.

### Data Augmentation

**Data augmentation** artificially increases training set size by generating realistic variants of training images.

**Common transformations**:
- Shift (translate horizontally and vertically)
- Rotate
- Resize/zoom
- Flip horizontally (but not vertically for most datasets)
- Adjust contrast and brightness
- Change color balance

**Goal**: Force model to be tolerant to variations while keeping transformations realistic.

Let's implement data augmentation for our Fashion MNIST model!

In [ ]:
# Data Augmentation in Keras
data_augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.1),  # Rotate by up to 10% (36 degrees)
    keras.layers.RandomZoom(0.1),  # Zoom by up to 10%
    keras.layers.RandomTranslation(0.1, 0.1),  # Shift by up to 10%
], name="data_augmentation")

# Visualize augmented images
sample_image = X_train[0:1]

plt.figure(figsize=(15, 10))
plt.suptitle('Data Augmentation Examples', fontsize=16)

# Original image
plt.subplot(3, 5, 1)
plt.imshow(sample_image[0, :, :, 0], cmap='gray')
plt.title('Original', fontsize=10)
plt.axis('off')

# Generate 14 augmented versions
for i in range(14):
    augmented = data_augmentation(sample_image, training=True)
    plt.subplot(3, 5, i + 2)
    plt.imshow(augmented[0, :, :, 0], cmap='gray')
    plt.title(f'Augmented {i+1}', fontsize=10)
    plt.axis('off')

plt.tight_layout()
plt.show()

print("Notice the variations:")
print("  - Horizontal flips")
print("  - Rotations")
print("  - Zooming in/out")
print("  - Small translations")
print("\nThese variations help the model generalize better!")

### ResNet (2015)

**Won ILSVRC 2015** with <3.6% top-5 error rate - better than human performance!

**Key Innovation: Skip Connections (Residual Connections)**

Instead of learning $h(x)$, the network learns the residual $f(x) = h(x) - x$.

**Benefits**:
1. When initialized, network outputs ~0, so with skip connections it models identity function initially
2. Speeds up training if target function is close to identity
3. **Signal easily propagates** through entire network (solves vanishing gradient problem)
4. Network can make progress even if some layers haven't started learning

**Architecture variants**:
- ResNet-34: 34 layers
- ResNet-50: 50 layers (uses bottleneck residual units)
- ResNet-101: 101 layers
- ResNet-152: 152 layers (winning variant)

### Residual Unit Structure

Standard residual unit:
1. 3×3 convolution + Batch Normalization + ReLU
2. 3×3 convolution + Batch Normalization
3. Add input (skip connection)
4. ReLU

Bottleneck residual unit (for deeper networks):
1. 1×1 convolution (reduce channels by 4)
2. 3×3 convolution
3. 1×1 convolution (restore channels)
4. Add input (skip connection)
5. ReLU

In [ ]:
# Implement a Residual Unit
class ResidualUnit(keras.layers.Layer):
    def __init__(self, filters, strides=1, activation="relu", **kwargs):
        super().__init__(**kwargs)
        self.activation = keras.activations.get(activation)
        self.main_layers = [
            keras.layers.Conv2D(filters, 3, strides=strides,
                              padding="same", use_bias=False),
            keras.layers.BatchNormalization(),
            self.activation,
            keras.layers.Conv2D(filters, 3, strides=1,
                              padding="same", use_bias=False),
            keras.layers.BatchNormalization()
        ]
        self.skip_layers = []
        if strides > 1:
            self.skip_layers = [
                keras.layers.Conv2D(filters, 1, strides=strides,
                                  padding="same", use_bias=False),
                keras.layers.BatchNormalization()
            ]
    
    def call(self, inputs):
        Z = inputs
        for layer in self.main_layers:
            Z = layer(Z)
        skip_Z = inputs
        for layer in self.skip_layers:
            skip_Z = layer(skip_Z)
        return self.activation(Z + skip_Z)

# Build a simple ResNet-like model for Fashion MNIST
def create_simple_resnet():
    model = keras.models.Sequential([
        keras.layers.Conv2D(64, 7, strides=2, input_shape=[28, 28, 1],
                          padding="same", use_bias=False),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.MaxPool2D(pool_size=3, strides=2, padding="same"),
    ], name='SimpleResNet')
    
    # Add residual units
    prev_filters = 64
    for filters in [64, 64, 128, 128, 256, 256]:
        strides = 1 if filters == prev_filters else 2
        model.add(ResidualUnit(filters, strides=strides))
        prev_filters = filters
    
    model.add(keras.layers.GlobalAvgPool2D())
    model.add(keras.layers.Flatten())
    model.add(keras.layers.Dense(10, activation="softmax"))
    
    return model

simple_resnet = create_simple_resnet()
simple_resnet.summary()

print(f"\nTotal parameters: {simple_resnet.count_params():,}")
print("\nKey features:")
print("  - Skip connections allow gradient to flow directly")
print("  - Batch Normalization after each convolution")
print("  - Global Average Pooling instead of Flatten + Dense")
print("  - Can train very deep networks without vanishing gradients")

### Other Notable Architectures

#### VGGNet (2014)
- Runner-up in ILSVRC 2014
- **Very simple architecture**: 2-3 convolutional layers → pooling (repeated)
- **Only 3×3 filters**, but many of them
- 16 or 19 convolutional layers total
- Classical, straightforward design

#### GoogLeNet / Inception (2014)
- **Won ILSVRC 2014** with <7% top-5 error
- **Innovation: Inception Modules**
- Processes input through 4 parallel paths:
  1. 1×1 convolution
  2. 1×1 → 3×3 convolution
  3. 1×1 → 5×5 convolution
  4. 3×3 max pooling → 1×1 convolution
- **1×1 convolutions** act as bottleneck layers to reduce dimensionality
- ~6M parameters (10× fewer than AlexNet!)

#### Xception (2016)
- Created by François Chollet (Keras author)
- **Innovation: Depthwise Separable Convolutions**
- Separates spatial patterns from cross-channel patterns:
  1. **Depthwise convolution**: One spatial filter per input channel
  2. **Pointwise convolution**: 1×1 convolution for cross-channel patterns
- Benefits: Fewer parameters, less memory, often better performance

#### SENet (2017)
- **Won ILSVRC 2017** with 2.25% top-5 error
- **Innovation: SE Blocks (Squeeze-and-Excitation)**
- SE block analyzes layer output focusing on depth dimension
- Learns which features typically active together
- Recalibrates feature maps accordingly
- Can extend existing architectures (SE-Inception, SE-ResNet)

## Using Pretrained Models

Keras provides pretrained models in `keras.applications` that were trained on ImageNet (1.4M images, 1000 classes).

**Available models**:
- ResNet variants (ResNet50, ResNet101, ResNet152, ResNet50V2, ResNet101V2, ResNet152V2)
- Inception variants (InceptionV3, InceptionResNetV2)
- VGGNet (VGG16, VGG19)
- Xception
- MobileNet, MobileNetV2 (optimized for mobile devices)
- EfficientNet variants
- And more!

Let's see how to use a pretrained model for image classification.

In [ ]:
# Load a pretrained ResNet50 model
print("Loading pretrained ResNet50...")
pretrained_model = keras.applications.resnet50.ResNet50(weights="imagenet")

print("Model loaded successfully!")
print(f"\nModel summary:")
print(f"  Input shape: (224, 224, 3)")
print(f"  Output shape: (1000,) - ImageNet classes")
print(f"  Total parameters: {pretrained_model.count_params():,}")

# Using the sample images we loaded earlier
print("\n" + "="*60)
print("Classifying sample images...")

# Resize images to expected size (224×224)
images_resized = tf.image.resize(images, [224, 224])

# Preprocess (each model has specific requirements)
inputs = keras.applications.resnet50.preprocess_input(images_resized * 255)

# Make predictions
print("Running inference...")
Y_proba = pretrained_model.predict(inputs, verbose=0)
print(f"Predictions shape: {Y_proba.shape}")

# Decode predictions
top_K = keras.applications.resnet50.decode_predictions(Y_proba, top=3)

# Display results
for image_idx, image_name in enumerate(['China', 'Flower']):
    print(f"\n{image_name} Image - Top 3 Predictions:")
    for class_id, class_name, score in top_K[image_idx]:
        print(f"  {class_name:20s}: {score*100:5.2f}%")

print("\n" + "="*60)
print("The model correctly identifies objects in both images!")

## Transfer Learning

**Transfer learning** allows us to reuse pretrained models for new tasks. This is extremely powerful when:
- You have limited training data
- Training from scratch would take too long
- You want to leverage knowledge learned from large datasets

**Approach**:
1. Load a pretrained model (trained on ImageNet)
2. Remove the top layers (classification head)
3. Add new layers for your specific task
4. **Freeze** pretrained layers initially (don't train them)
5. Train only the new layers
6. Optionally **unfreeze** and fine-tune the entire network

Let's demonstrate transfer learning by creating a flower classifier!

In [ ]:
# Create a transfer learning model
# We'll use a smaller pretrained model for demonstration

# Load MobileNetV2 without top layer (include_top=False)
base_model = keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,  # Don't include the classification head
    weights='imagenet'
)

print("Base Model (MobileNetV2) loaded")
print(f"  Parameters: {base_model.count_params():,}")
print(f"  Output shape: {base_model.output.shape}")

# Freeze the base model
base_model.trainable = False
print("  Base model frozen (not trainable)")

# Build new model with custom top layers
inputs = keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)  # Base model in inference mode
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dense(128, activation='relu')(x)
x = keras.layers.Dropout(0.5)(x)
outputs = keras.layers.Dense(5, activation='softmax')(x)  # 5 classes for flowers

transfer_model = keras.Model(inputs, outputs, name='FlowerClassifier')

print("\nTransfer Learning Model:")
transfer_model.summary()

print("\n" + "="*70)
print("Transfer Learning Workflow:")
print("  1. ✓ Load pretrained base model (MobileNetV2)")
print("  2. ✓ Remove top layer (include_top=False)")
print("  3. ✓ Add custom layers for flower classification (5 classes)")
print("  4. ✓ Freeze base model layers")
print("  5. → Train only new layers (fast, prevents destroying pretrained weights)")
print("  6. → Optionally unfreeze and fine-tune (for better accuracy)")

# Count trainable vs non-trainable parameters
trainable_params = sum([np.prod(v.shape) for v in transfer_model.trainable_weights])
non_trainable_params = sum([np.prod(v.shape) for v in transfer_model.non_trainable_weights])

print(f"\nParameter breakdown:")
print(f"  Trainable: {trainable_params:,} (only new layers)")
print(f"  Non-trainable: {non_trainable_params:,} (frozen base model)")
print(f"  Total: {trainable_params + non_trainable_params:,}")

## Advanced Concepts and Applications

### Object Detection

**Object detection** involves detecting and localizing multiple objects in an image.

**Popular architectures**:

1. **YOLO (You Only Look Once)** - Extremely fast, real-time detection
   - Divides image into grid
   - Each grid cell predicts multiple bounding boxes
   - Single forward pass through network
   - Versions: YOLOv1, YOLOv2, YOLOv3, YOLOv4, YOLOv5, YOLO9000

2. **Faster R-CNN** - High accuracy
   - Region Proposal Network (RPN) suggests regions
   - Classifier for each proposed region
   - More accurate but slower than YOLO

3. **SSD (Single Shot Detector)** - Balance of speed and accuracy
   - Similar to YOLO
   - Multi-scale feature maps

**Evaluation metric: Mean Average Precision (mAP)**
- mAP@0.5: Uses IoU threshold of 0.5
- mAP@[.50:.95]: Averages over IoU thresholds from 0.50 to 0.95

### Semantic Segmentation

**Semantic segmentation** classifies each pixel according to the object class it belongs to.

**Challenge**: CNNs reduce spatial resolution (due to strides), losing precise location information.

**Solutions**:
1. **Transposed Convolutions** (deconvolution) for upsampling
2. **Skip Connections** to recover spatial information from earlier layers
3. **U-Net architecture** - Popular for medical image segmentation

### Instance Segmentation

**Instance segmentation** distinguishes individual objects (not merging objects of same class).

**Mask R-CNN**: Extends Faster R-CNN by adding pixel mask prediction for each bounding box.

In [ ]:
# Demonstrate depthwise separable convolution
print("Depthwise Separable Convolution")
print("="*60)

# Standard convolution parameters
input_channels = 64
output_channels = 128
kernel_size = 3
feature_map_size = 56

# Standard convolution
standard_params = kernel_size * kernel_size * input_channels * output_channels
standard_ops = standard_params * feature_map_size * feature_map_size

print("Standard Convolution:")
print(f"  Parameters: {standard_params:,}")
print(f"  Operations: {standard_ops:,}")

# Depthwise separable convolution
# 1. Depthwise: one kernel per input channel
depthwise_params = kernel_size * kernel_size * input_channels
# 2. Pointwise: 1x1 convolution across channels
pointwise_params = 1 * 1 * input_channels * output_channels
separable_params = depthwise_params + pointwise_params
separable_ops = (depthwise_params + pointwise_params) * feature_map_size * feature_map_size

print("\nDepthwise Separable Convolution:")
print(f"  Depthwise parameters: {depthwise_params:,}")
print(f"  Pointwise parameters: {pointwise_params:,}")
print(f"  Total parameters: {separable_params:,}")
print(f"  Operations: {separable_ops:,}")

print("\nReduction:")
print(f"  Parameters: {standard_params / separable_params:.2f}x fewer")
print(f"  Operations: {standard_ops / separable_ops:.2f}x fewer")

print("\nKeras implementation:")
print("  keras.layers.SeparableConv2D(filters, kernel_size, ...)")

# Create a comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

categories = ['Standard\nConvolution', 'Depthwise\nSeparable']
params = [standard_params, separable_params]
ops = [standard_ops / 1e6, separable_ops / 1e6]

axes[0].bar(categories, params, color=['#e74c3c', '#2ecc71'])
axes[0].set_ylabel('Parameters')
axes[0].set_title('Parameter Comparison')
axes[0].set_ylim([0, max(params) * 1.1])
for i, v in enumerate(params):
    axes[0].text(i, v + max(params) * 0.02, f'{v:,}', ha='center', fontsize=11)

axes[1].bar(categories, ops, color=['#e74c3c', '#2ecc71'])
axes[1].set_ylabel('Operations (millions)')
axes[1].set_title('Computational Comparison')
axes[1].set_ylim([0, max(ops) * 1.1])
for i, v in enumerate(ops):
    axes[1].text(i, v + max(ops) * 0.02, f'{v:.1f}M', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

## Practical Tips and Best Practices

### Model Design

1. **Start simple, then increase complexity**
   - Begin with a small model
   - Add layers/filters gradually
   - Monitor training and validation metrics

2. **Use proven architectures**
   - Start with established architectures (ResNet, EfficientNet)
   - Modify for your specific use case
   - Don't reinvent the wheel

3. **Leverage transfer learning**
   - Almost always better than training from scratch
   - Especially with limited data
   - Fine-tune for best results

### Training Tips

1. **Data augmentation is crucial**
   - Prevents overfitting
   - Improves generalization
   - Use appropriate transformations for your domain

2. **Use batch normalization**
   - Speeds up training
   - Acts as regularization
   - Place after convolution, before activation

3. **Monitor overfitting**
   - Watch validation metrics
   - Use dropout in dense layers
   - Use early stopping

4. **Learning rate scheduling**
   - Start with higher learning rate
   - Reduce when validation loss plateaus
   - Use callbacks like `ReduceLROnPlateau`

### Computational Efficiency

1. **Memory management**
   - Reduce batch size if OOM
   - Use mixed precision training (float16)
   - Clear unused variables

2. **Use appropriate stride**
   - Stride 2 instead of pooling sometimes
   - Reduces computation while downsampling

3. **Global Average Pooling**
   - Replace Flatten + Dense with GlobalAvgPool + Dense
   - Fewer parameters
   - Works with variable input sizes

## Complete Summary

### Part 1: Foundations (Introduction & Convolutional Layers)
✓ CNN origins from visual cortex research  
✓ Convolutional layers with local receptive fields  
✓ Parameter sharing and translation invariance  
✓ Filters and feature maps  
✓ Padding and stride concepts  
✓ Memory requirements and optimization  

### Part 2: Building Blocks (Pooling & Architectures)
✓ Max and average pooling  
✓ Translation invariance benefits  
✓ CNN architecture patterns  
✓ Fashion MNIST implementation  
✓ Visualizing learned features  
✓ Feature hierarchy in deep networks  

### Part 3: Advanced Topics
✓ **Historic architectures**: LeNet-5, AlexNet  
✓ **Modern architectures**: ResNet, VGGNet, GoogLeNet, Xception, SENet  
✓ **Skip connections**: Solving vanishing gradients  
✓ **Data augmentation**: Improving generalization  
✓ **Pretrained models**: Using keras.applications  
✓ **Transfer learning**: Reusing learned features  
✓ **Advanced applications**: Object detection, semantic segmentation  
✓ **Optimization techniques**: Depthwise separable convolutions  

## Key Takeaways

### 1. CNNs Are Hierarchical
- Early layers: Simple features (edges, textures)
- Middle layers: Patterns and shapes
- Deep layers: Complex, abstract concepts
- This mirrors how the human visual cortex works!

### 2. Design Principles
- **Convolutional layers**: Feature extraction
- **Pooling layers**: Dimensionality reduction
- **Skip connections**: Enable very deep networks
- **Batch normalization**: Faster, more stable training
- **Data augmentation**: Prevent overfitting

### 3. Transfer Learning is Powerful
- Leverage models trained on millions of images
- Fine-tune for your specific task
- Achieve great results with limited data
- Much faster than training from scratch

### 4. Architecture Evolution
- **LeNet-5 (1998)**: Proved CNNs work
- **AlexNet (2012)**: Sparked the deep learning revolution
- **VGGNet (2014)**: Showed simplicity can work
- **GoogLeNet (2014)**: Efficient with inception modules
- **ResNet (2015)**: Enabled very deep networks
- **Xception (2016)**: Improved efficiency with separable convolutions
- **SENet (2017)**: Channel-wise recalibration

### 5. Practical Applications
- **Image classification**: Categorizing entire images
- **Object detection**: Finding and localizing objects
- **Semantic segmentation**: Pixel-level classification
- **Instance segmentation**: Distinguishing individual objects
- **Face recognition**, **medical imaging**, **autonomous vehicles**, and more!

## What's Next?

To deepen your understanding:
1. **Train models** on your own datasets
2. **Experiment** with different architectures
3. **Study recent papers** (EfficientNet, Vision Transformers)
4. **Participate in Kaggle competitions**
5. **Explore specialized applications** (medical imaging, satellite imagery)

The field continues to evolve rapidly with new architectures, training techniques, and applications emerging regularly!

In [ ]:
# Final Quick Reference
print("="*70)
print("QUICK REFERENCE: CNN Implementation in Keras")
print("="*70)

print("\n1. BASIC CNN LAYER")
print("-" * 70)
print("""
keras.layers.Conv2D(
    filters=32,        # Number of filters
    kernel_size=3,     # 3x3 kernel
    strides=1,         # Stride
    padding='same',    # Padding type
    activation='relu'  # Activation function
)
""")

print("2. POOLING LAYERS")
print("-" * 70)
print("""
keras.layers.MaxPool2D(pool_size=2)
keras.layers.AvgPool2D(pool_size=2)
keras.layers.GlobalAvgPool2D()
""")

print("3. BATCH NORMALIZATION")
print("-" * 70)
print("""
keras.layers.BatchNormalization()
""")

print("4. DATA AUGMENTATION")
print("-" * 70)
print("""
keras.Sequential([
    keras.layers.RandomFlip('horizontal'),
    keras.layers.RandomRotation(0.1),
    keras.layers.RandomZoom(0.1),
    keras.layers.RandomTranslation(0.1, 0.1)
])
""")

print("5. TRANSFER LEARNING")
print("-" * 70)
print("""
# Load base model
base_model = keras.applications.MobileNetV2(
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze

# Add custom layers
x = base_model.output
x = keras.layers.GlobalAvgPool2D()(x)
x = keras.layers.Dense(128, activation='relu')(x)
outputs = keras.layers.Dense(num_classes, activation='softmax')(x)
model = keras.Model(base_model.input, outputs)
""")

print("6. RESIDUAL CONNECTIONS")
print("-" * 70)
print("""
# In custom layer
def call(self, inputs):
    x = self.conv1(inputs)
    x = self.conv2(x)
    return self.activation(x + inputs)  # Skip connection
""")

print("\n" + "="*70)
print("Thank you for exploring CNNs with this notebook!")
print("="*70)

---

# Part 4: Advanced Applications

## Classification and Localization

**Classification**: Identify what object is in an image  
**Localization**: Determine where the object is located

To add localization, we add a **regression task** that predicts bounding box coordinates.

### Bounding Box Representation

A bounding box is typically represented by 4 values:
- **center_x**: Horizontal center position (0-1 normalized)
- **center_y**: Vertical center position (0-1 normalized)
- **height**: Box height (0-1 normalized)
- **width**: Box width (0-1 normalized)

Alternative: Sometimes we predict $\sqrt{height}$ and $\sqrt{width}$ instead of raw dimensions for better numerical stability.

In [ ]:
# Demonstrate classification + localization model architecture
print("Classification and Localization Model")
print("="*70)

# Create a model with two outputs: class and bounding box
def create_classification_localization_model(n_classes=10):
    """
    Model with two outputs:
    1. Class probabilities
    2. Bounding box coordinates
    """
    base_model = keras.applications.MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False
    
    # Shared features
    x = keras.layers.GlobalAveragePooling2D()(base_model.output)
    x = keras.layers.Dense(256, activation='relu')(x)
    x = keras.layers.Dropout(0.5)(x)
    
    # Classification head
    class_output = keras.layers.Dense(n_classes, activation='softmax', 
                                     name='class_output')(x)
    
    # Localization head (4 values: center_x, center_y, height, width)
    loc_output = keras.layers.Dense(4, activation='sigmoid',
                                   name='loc_output')(x)  # Sigmoid to keep 0-1
    
    model = keras.Model(inputs=base_model.input, 
                       outputs=[class_output, loc_output],
                       name='ClassificationLocalization')
    
    return model

clf_loc_model = create_classification_localization_model(n_classes=5)

print("Model Architecture:")
print(f"  Input: RGB image (224x224x3)")
print(f"  Base: MobileNetV2 (frozen, pretrained)")
print(f"  Shared: GlobalAvgPool + Dense(256) + Dropout")
print(f"  Output 1: Class probabilities (5 classes)")
print(f"  Output 2: Bounding box (4 coordinates)")

print("\nCompiling with multiple losses:")
clf_loc_model.compile(
    loss={
        'class_output': 'sparse_categorical_crossentropy',
        'loc_output': 'mse'  # Mean Squared Error for bbox coordinates
    },
    loss_weights={
        'class_output': 0.8,  # Classification is more important
        'loc_output': 0.2
    },
    optimizer='adam',
    metrics={
        'class_output': 'accuracy'
    }
)

print("  Classification loss: Sparse Categorical Crossentropy (weight=0.8)")
print("  Localization loss: MSE (weight=0.2)")
print("\nNote: Loss weights can be tuned based on task requirements")

### Intersection over Union (IoU)

**IoU** is the standard metric for evaluating bounding box predictions.

$$\text{IoU} = \frac{\text{Area of Overlap}}{\text{Area of Union}}$$

- **IoU = 1.0**: Perfect overlap
- **IoU = 0.5**: Decent localization
- **IoU = 0.0**: No overlap

Keras provides: `tf.keras.metrics.MeanIoU`

In [ ]:
# Implement IoU calculation
def calculate_iou(box1, box2):
    """
    Calculate Intersection over Union for two bounding boxes
    
    Args:
        box1, box2: [x_min, y_min, x_max, y_max]
    
    Returns:
        IoU score (0 to 1)
    """
    # Calculate intersection coordinates
    x_left = max(box1[0], box2[0])
    y_top = max(box1[1], box2[1])
    x_right = min(box1[2], box2[2])
    y_bottom = min(box1[3], box2[3])
    
    # Check if there's an intersection
    if x_right < x_left or y_bottom < y_top:
        return 0.0
    
    # Calculate areas
    intersection_area = (x_right - x_left) * (y_bottom - y_top)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = box1_area + box2_area - intersection_area
    
    iou = intersection_area / union_area
    return iou

# Visualize IoU examples
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

examples = [
    {
        'box1': [10, 10, 50, 50],
        'box2': [30, 30, 70, 70],
        'title': 'Moderate Overlap'
    },
    {
        'box1': [10, 10, 50, 50],
        'box2': [15, 15, 45, 45],
        'title': 'High Overlap'
    },
    {
        'box1': [10, 10, 40, 40],
        'box2': [50, 50, 80, 80],
        'title': 'No Overlap'
    }
]

for idx, example in enumerate(examples):
    ax = axes[idx]
    box1 = example['box1']
    box2 = example['box2']
    
    iou = calculate_iou(box1, box2)
    
    # Draw boxes
    from matplotlib.patches import Rectangle
    rect1 = Rectangle((box1[0], box1[1]), box1[2]-box1[0], box1[3]-box1[1],
                     linewidth=2, edgecolor='blue', facecolor='blue', alpha=0.3)
    rect2 = Rectangle((box2[0], box2[1]), box2[2]-box2[0], box2[3]-box2[1],
                     linewidth=2, edgecolor='red', facecolor='red', alpha=0.3)
    
    ax.add_patch(rect1)
    ax.add_patch(rect2)
    ax.set_xlim([0, 100])
    ax.set_ylim([0, 100])
    ax.set_aspect('equal')
    ax.set_title(f"{example['title']}\nIoU = {iou:.3f}", fontsize=12)
    ax.legend(['Ground Truth', 'Prediction'])

plt.tight_layout()
plt.show()

print("IoU Interpretation:")
print("  IoU > 0.7: Excellent localization")
print("  IoU > 0.5: Good localization (common threshold)")
print("  IoU > 0.3: Fair localization")
print("  IoU < 0.3: Poor localization")

## Semantic Segmentation

**Semantic segmentation** classifies each pixel in an image. Unlike object detection, it provides pixel-level understanding.

**Challenge**: CNNs reduce spatial resolution through pooling and strided convolutions, losing precise location information.

**Solution**: Use transposed convolutions (also called deconvolutions) to upsample feature maps back to original resolution.

### Transposed Convolution

A transposed convolution is the reverse of a regular convolution:
- Regular convolution: Input → smaller output
- Transposed convolution: Input → larger output

**How it works**: Insert empty rows/columns between input pixels, then apply regular convolution.

**Important**: In transposed convolution, **larger stride = larger output** (opposite of regular convolution!)

### Skip Connections for Better Resolution

To recover spatial information lost during downsampling:
1. Take feature maps from earlier layers (before pooling)
2. Upsample current layer
3. Concatenate or add with earlier feature maps
4. Continue upsampling

This is the key idea behind **U-Net**, a popular architecture for medical image segmentation.

In [ ]:
# Demonstrate transposed convolution
print("Transposed Convolution Demonstration")
print("="*70)

# Create a simple input
simple_input = np.array([[1, 2], [3, 4]], dtype=np.float32).reshape(1, 2, 2, 1)

print("Input (2x2):")
print(simple_input[0, :, :, 0])

# Regular convolution (downsampling)
regular_conv = keras.layers.Conv2D(1, kernel_size=2, strides=2, padding='valid')
downsampled = regular_conv(simple_input)

print(f"\nAfter regular Conv2D (stride=2): {downsampled.shape}")
print(f"  Output: {downsampled.shape[1]}x{downsampled.shape[2]} (downsampled)")

# Transposed convolution (upsampling)
transposed_conv = keras.layers.Conv2DTranspose(1, kernel_size=2, strides=2, padding='same')
upsampled = transposed_conv(simple_input)

print(f"\nAfter Conv2DTranspose (stride=2): {upsampled.shape}")
print(f"  Output: {upsampled.shape[1]}x{upsampled.shape[2]} (upsampled)")

# Demonstrate with larger example
print("\n" + "="*70)
print("Upsampling Example: 8x8 → 16x16")

# Create 8x8 feature map
feature_map = np.random.rand(1, 8, 8, 16)
print(f"Input shape: {feature_map.shape}")

# Upsample 2x
upsample_layer = keras.layers.Conv2DTranspose(
    filters=32, 
    kernel_size=3, 
    strides=2,  # Stride=2 doubles spatial dimensions
    padding='same',
    activation='relu'
)
upsampled = upsample_layer(feature_map)
print(f"Output shape: {upsampled.shape}")
print(f"Spatial dimensions doubled: 8x8 → 16x16")

# Visualize the concept
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Original feature map
sample_fm = np.random.rand(8, 8)
axes[0].imshow(sample_fm, cmap='viridis', interpolation='nearest')
axes[0].set_title('Original Feature Map (8×8)', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Upsampled
upsampled_vis = np.random.rand(16, 16)
axes[1].imshow(upsampled_vis, cmap='viridis', interpolation='nearest')
axes[1].set_title('After Transposed Conv (16×16)', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Insight:")
print("  Regular Conv: Stride > 1 → Downsample")
print("  Transposed Conv: Stride > 1 → Upsample")

## Other TensorFlow Convolution Operations

Beyond 2D convolutions, TensorFlow supports various convolution types for different applications:

### 1. Conv1D - For Sequential Data
Used for time series, text, audio signals.

```python
keras.layers.Conv1D(filters=64, kernel_size=3, ...)
```

### 2. Conv3D - For Volumetric Data  
Used for 3D medical scans (CT, MRI), video processing.

```python
keras.layers.Conv3D(filters=32, kernel_size=3, ...)
```

### 3. Dilated (Atrous) Convolution
Increases receptive field without increasing parameters by inserting "holes" in the filter.

```python
keras.layers.Conv2D(filters=64, kernel_size=3, dilation_rate=2, ...)
```

### 4. Depthwise Convolution
Applies each filter to each channel independently (used in separable convolutions).

```python
tf.nn.depthwise_conv2d(...)
```

In [ ]:
# Demonstrate different convolution types
print("Various Convolution Types in TensorFlow/Keras")
print("="*70)

# 1. Conv1D example
print("\n1. Conv1D - For Sequential Data (e.g., time series, text)")
print("-"*70)
sequence_data = np.random.rand(1, 100, 16)  # 1 sample, 100 timesteps, 16 features
conv1d = keras.layers.Conv1D(filters=32, kernel_size=5, padding='same')
output_1d = conv1d(sequence_data)
print(f"Input shape: {sequence_data.shape}")
print(f"Output shape: {output_1d.shape}")
print(f"Use cases: Text classification, audio processing, sensor data")

# 2. Conv3D example
print("\n2. Conv3D - For Volumetric Data (3D)")
print("-"*70)
volume_data = np.random.rand(1, 32, 32, 32, 1)  # 1 sample, 32x32x32 volume, 1 channel
conv3d = keras.layers.Conv3D(filters=16, kernel_size=3, padding='same')
output_3d = conv3d(volume_data)
print(f"Input shape: {volume_data.shape}")
print(f"Output shape: {output_3d.shape}")
print(f"Use cases: Medical imaging (CT/MRI), video processing, 3D modeling")

# 3. Dilated convolution example
print("\n3. Dilated (Atrous) Convolution - Expanded Receptive Field")
print("-"*70)
image_2d = np.random.rand(1, 64, 64, 3)
conv_dilated = keras.layers.Conv2D(filters=32, kernel_size=3, dilation_rate=2, padding='same')
output_dilated = conv_dilated(image_2d)
print(f"Input shape: {image_2d.shape}")
print(f"Output shape: {output_dilated.shape}")
print(f"Dilation rate=2: Receptive field is 3 + (3-1)*2 = 7×7")
print(f"Use cases: Semantic segmentation, capturing multi-scale context")

# Visualize receptive field comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Regular 3x3 convolution
regular_field = np.zeros((7, 7))
regular_field[2:5, 2:5] = 1
axes[0].imshow(regular_field, cmap='Blues', interpolation='nearest')
axes[0].set_title('Regular 3×3 Conv\nReceptive Field', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Dilated 3x3 convolution with rate=2
dilated_field = np.zeros((7, 7))
dilated_field[1, 1] = 1; dilated_field[1, 3] = 1; dilated_field[1, 5] = 1
dilated_field[3, 1] = 1; dilated_field[3, 3] = 1; dilated_field[3, 5] = 1
dilated_field[5, 1] = 1; dilated_field[5, 3] = 1; dilated_field[5, 5] = 1
axes[1].imshow(dilated_field, cmap='Reds', interpolation='nearest')
axes[1].set_title('Dilated 3×3 Conv (rate=2)\nReceptive Field (7×7 coverage)', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n4. Separable Convolution Summary")
print("-"*70)
print("Depthwise: One filter per channel (spatial patterns)")
print("Pointwise: 1×1 conv across channels (channel patterns)")
print("Benefit: ~8-9× fewer parameters and computations")
print("Implementation: keras.layers.SeparableConv2D(...)")

## Practice Exercises

Now that you've learned about CNNs, try these exercises to deepen your understanding!

### Exercise 1: CNN Advantages
**Question**: What advantages do CNNs have over fully connected DNNs for image classification?

**Answer**:
1. **Parameter Efficiency**: Local connectivity and weight sharing reduce parameters dramatically
2. **Translation Invariance**: Same filter applied across entire image
3. **Hierarchical Feature Learning**: Low-level → high-level features
4. **Spatial Structure Preservation**: 2D structure maintained throughout network
5. **Faster Training**: Fewer parameters to optimize

### Exercise 2: Parameter Calculation
**Question**: For a CNN with three 3×3 convolutional layers (stride 2, "same" padding) outputting 100, 200, and 400 feature maps on 200×300 RGB images, calculate total parameters and RAM requirements.

Let's solve this step by step!

In [ ]:
# Exercise 2 Solution: Calculate parameters and memory
print("Exercise 2: Parameter and Memory Calculation")
print("="*70)

# Given information
input_h, input_w, input_channels = 200, 300, 3
kernel_size = 3
stride = 2
layers = [
    {'filters': 100, 'input_channels': 3},
    {'filters': 200, 'input_channels': 100},
    {'filters': 400, 'input_channels': 200}
]

print("Configuration:")
print(f"  Input: {input_h}×{input_w} RGB image")
print(f"  3 Conv layers: 3×3 kernel, stride 2, 'same' padding")
print(f"  Filters: 100, 200, 400\n")

total_params = 0
current_h, current_w = input_h, input_w

print("Layer-by-layer analysis:")
print("-"*70)

for i, layer in enumerate(layers, 1):
    filters = layer['filters']
    in_channels = layer['input_channels']
    
    # Calculate parameters
    params_per_filter = (kernel_size * kernel_size * in_channels) + 1  # +1 for bias
    layer_params = params_per_filter * filters
    total_params += layer_params
    
    # Calculate output dimensions (with stride 2)
    current_h = (current_h + stride - 1) // stride  # Ceiling division
    current_w = (current_w + stride - 1) // stride
    
    # Calculate memory for this layer
    neurons = current_h * current_w * filters
    memory_mb = neurons * 4 / (1024**2)  # 4 bytes per float32
    
    print(f"Layer {i}:")
    print(f"  Input: {in_channels} channels")
    print(f"  Filters: {filters}")
    print(f"  Parameters: {layer_params:,} = ({kernel_size}×{kernel_size}×{in_channels}+1)×{filters}")
    print(f"  Output: {current_h}×{current_w}×{filters}")
    print(f"  Memory (per image): {memory_mb:.2f} MB")
    print()

print("="*70)
print(f"TOTAL PARAMETERS: {total_params:,}")

# Memory for batch of 100
batch_size = 100
batch_memory = 0
current_h, current_w = input_h, input_w

for layer in layers:
    current_h = (current_h + stride - 1) // stride
    current_w = (current_w + stride - 1) // stride
    neurons = current_h * current_w * layer['filters']
    batch_memory += neurons * 4 * batch_size / (1024**2)

print(f"MEMORY (batch of {batch_size}): {batch_memory:.2f} MB")
print("="*70)

### Exercise 3-11: Quick Answers

**Exercise 3: GPU Out of Memory - What to do?**
1. Reduce mini-batch size
2. Increase stride (reduce spatial dimensions)
3. Remove layers or reduce filters
4. Use 16-bit floats instead of 32-bit
5. Distribute training across multiple GPUs

**Exercise 4: Why max pooling vs convolutional layer with same stride?**
- Max pooling has **no parameters** (no learning required)
- Provides **translation invariance**
- Acts as a form of **regularization**
- **Computationally cheaper** than convolution
- Preserves **strongest activations**

**Exercise 5: When to use Local Response Normalization (LRN)?**
- **Rarely used today** - largely replaced by Batch Normalization
- Historical significance: Used in AlexNet (2012)
- BN is more effective and has become the standard

**Exercise 6: Main innovations in famous architectures**
- **AlexNet (2012)**: ReLU, dropout, data augmentation, GPU training
- **GoogLeNet (2014)**: Inception modules, 1×1 convolutions, fewer parameters
- **ResNet (2015)**: Skip connections, very deep networks, batch normalization
- **SENet (2017)**: Squeeze-and-Excitation blocks, channel recalibration
- **Xception (2016)**: Depthwise separable convolutions, efficiency

**Exercise 7: What is a Fully Convolutional Network (FCN)?**
- CNN where **dense layers are replaced** with convolutional layers
- Can process **images of any size** (not fixed input size)
- **Conversion**: Dense(N neurons) → Conv2D(N filters, kernel=input_size, padding='valid')
- Used in: Semantic segmentation, object detection

**Exercise 8: Main difficulty of semantic segmentation?**
- **Loss of spatial resolution** due to pooling and strided convolutions
- **Solution**: Transposed convolutions + skip connections (U-Net architecture)
- Need to balance receptive field size with precise localization

## Final Comprehensive Summary

### What We've Covered

This notebook provided a complete, hands-on exploration of Convolutional Neural Networks:

#### Part 1: Foundations (25%)
✅ CNN origins and motivation  
✅ Visual cortex architecture inspiration  
✅ Convolutional layers with receptive fields  
✅ Filters and feature maps  
✅ Translation invariance through weight sharing  
✅ Memory requirements and optimization strategies  

#### Part 2: Building Blocks (25%)
✅ Max pooling and average pooling  
✅ Translation invariance benefits  
✅ CNN architecture patterns  
✅ Complete Fashion MNIST implementation  
✅ Filter and feature map visualization  
✅ Hierarchical feature learning  

#### Part 3: Advanced Architectures (25%)
✅ LeNet-5: The pioneer  
✅ AlexNet: Deep learning revolution  
✅ ResNet: Skip connections solution  
✅ VGGNet, GoogLeNet, Xception, SENet  
✅ Pretrained models and transfer learning  
✅ Data augmentation techniques  
✅ Depthwise separable convolutions  

#### Part 4: Applications & Beyond (25%)
✅ Classification and localization  
✅ Intersection over Union (IoU)  
✅ Semantic segmentation  
✅ Transposed convolutions  
✅ Conv1D, Conv3D, dilated convolutions  
✅ Practice exercises with solutions  

### Key Insights

1. **CNNs revolutionized computer vision** by mimicking the human visual cortex
2. **Weight sharing** makes CNNs parameter-efficient and translation-invariant
3. **Hierarchical learning** enables detection of complex patterns from simple features
4. **Skip connections** (ResNet) solved the vanishing gradient problem
5. **Transfer learning** allows leveraging pre-trained models for new tasks
6. **Architecture evolution** shows continuous innovation: efficiency, depth, and performance

### Real-World Impact

CNNs power modern AI applications:
- **Autonomous vehicles**: Object detection and scene understanding
- **Medical imaging**: Disease detection and diagnosis
- **Face recognition**: Security and authentication
- **Image search**: Finding similar images
- **Content moderation**: Detecting inappropriate content
- **Augmented reality**: Scene understanding and tracking

### Next Steps for Learning

1. **Practice**: Build CNNs for different datasets (CIFAR-10, ImageNet)
2. **Experiment**: Try different architectures and hyperparameters
3. **Read papers**: Study recent architectures (EfficientNet, Vision Transformers)
4. **Kaggle**: Participate in computer vision competitions
5. **Specialize**: Explore specific domains (medical imaging, autonomous driving)
6. **Stay updated**: Follow latest research and techniques

### Resources for Further Learning

- **Papers**: arXiv.org for latest CNN research
- **Courses**: Fast.ai, deeplearning.ai, CS231n (Stanford)
- **Datasets**: ImageNet, COCO, Pascal VOC, Cityscapes
- **Frameworks**: TensorFlow, PyTorch, Keras
- **Communities**: Reddit r/MachineLearning, Papers with Code

---

## Thank You!

Congratulations on completing this comprehensive CNN tutorial! You now have:
- Strong theoretical foundation in CNNs
- Hands-on experience implementing CNNs
- Understanding of famous architectures
- Knowledge of advanced techniques
- Practical skills for real-world applications

**Keep learning, experimenting, and building amazing things with CNNs!** 🚀